# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Chạy trên Google Colab
1. Mở notebook trên Colab: **File → Open notebook → GitHub** → dán URL repo, hoặc upload file `.ipynb`.
2. Cell **1.0** tự `git clone` repo để có `data/` (Golden Dataset) và cấu trúc thư mục.
3. Tạo **Colab Secrets** (icon chiếc khóa 🔑 ở sidebar trái, nhớ bật *Notebook access* cho từng secret):

| Secret | Giá trị | Ghi chú |
|---|---|---|
| `NEO4J_URI` | `neo4j+s://<id>.databases.neo4j.io` | Tạo instance **AuraDB Free** tại console.neo4j.io |
| `NEO4J_USER` | `neo4j` | |
| `NEO4J_PASSWORD` | mật khẩu AuraDB | hiện ra khi tạo instance — lưu lại ngay |
| `OPENAI_API_KEY` | `sk-or-v1-...` | key OpenRouter (hoặc key OpenAI thường) |
| `OPENAI_BASE_URL` | `https://openrouter.ai/api/v1` | bỏ qua nếu dùng OpenAI trực tiếp |
| `EXTRACT_MODEL` | `openai/gpt-4o-mini` | model sinh câu trả lời & trích xuất KG |
| `JUDGE_PROVIDER` | `openai` | |
| `JUDGE_MODEL` | `google/gemini-2.5-flash` | model chấm điểm — cố ý KHÁC model sinh để giảm self-preference bias |
| `GROQ_API_KEY`, `GROQ_MODEL` | *(tùy chọn)* | nếu có key Groq còn hạn thì pipeline ưu tiên Groq |
| `HF_TOKEN` | *(tùy chọn)* | chỉ cần cho bản gốc HackerNoon (gated); không có sẽ tự stream mirror công khai `MongoDB/tech-news-embeddings` (cùng dữ liệu) |

> [!WARNING]
> Không hard-code API key hoặc mật khẩu Neo4j vào notebook nộp bài.


In [25]:
#@title 1.0 — (Colab) Clone repo để có data/ + cấu trúc thư mục (tự bỏ qua khi chạy local)
import os, subprocess

REPO_URL = "https://github.com/MinhCris/K3-Track3-Lab19-GraphRAG-2A202601459-DangQuangMinh.git"

if os.path.exists("data/graphrag_golden_50_first5000_detailed.csv"):
    print("✅ Đang ở trong repo (local/đã clone) — bỏ qua bước clone.")
else:
    if not os.path.exists("lab19"):
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, "lab19"])
    os.chdir("lab19")
    print("Đã clone repo · cwd ->", os.getcwd())


✅ Đang ở trong repo (local/đã clone) — bỏ qua bước clone.


In [26]:
#@title 1.1 — Install (tự bỏ qua nếu môi trường local đã cài đủ dependencies)
import importlib.util, subprocess, sys

_required = {
    "neo4j": "neo4j", "pandas": "pandas", "numpy": "numpy", "pyarrow": "pyarrow",
    "sentence_transformers": "sentence-transformers", "faiss": "faiss-cpu",
    "groq": "groq", "openai": "openai", "tqdm": "tqdm", "networkx": "networkx",
    "datasets": "datasets", "dotenv": "python-dotenv",
}
_missing = [pkg for mod, pkg in _required.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing:", _missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
else:
    print("✅ Dependencies đã sẵn sàng — bỏ qua bước cài đặt.")


✅ Dependencies đã sẵn sàng — bỏ qua bước cài đặt.


In [27]:
#@title 1.1b — Kaggle compatibility patch (chạy trước cell 1.2; vô hại trên Colab)
import os

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    _names = ["HF_TOKEN", "NEO4J_URI", "NEO4J_USER", "NEO4J_PASSWORD",
              "NEO4J_DATABASE", "GROQ_API_KEY", "GROQ_MODEL",
              "JUDGE_PROVIDER", "JUDGE_MODEL", "OPENAI_API_KEY",
              "OPENAI_BASE_URL", "EXTRACT_MODEL"]
    _loaded = []
    for _name in _names:
        try:
            _v = _secrets.get_secret(_name)
            if _v:
                os.environ[_name] = _v
                _loaded.append(_name)
        except Exception:
            pass
    print(f"Kaggle secrets -> os.environ: {_loaded}")
    _missing = [n for n in _names if n not in _loaded]
    if _missing:
        print(f"⚠️ Chưa có (thêm ở Add-ons -> Secrets rồi chạy lại): {_missing}")
except ImportError:
    print("Không phải môi trường Kaggle — bỏ qua patch.")

# Notebook dùng đường dẫn /content (kiểu Colab). Trên máy local (macOS/Linux)
# root filesystem có thể read-only -> bỏ qua, các cell sau đã ưu tiên đường dẫn repo.
try:
    os.makedirs("/content", exist_ok=True)
except OSError:
    pass


Không phải môi trường Kaggle — bỏ qua patch.


In [28]:
#@title 1.2 — Imports & config
import os
# Môi trường local có TensorFlow cũ (Keras 3) gây lỗi import transformers
# -> ép transformers chỉ dùng PyTorch backend. Vô hại trên Colab/Kaggle.
os.environ.setdefault("USE_TF", "0")
# macOS + anaconda: torch (libomp) và faiss/MKL (libiomp5) cùng nạp 2 OpenMP runtime
# -> kernel abort ngay op song song đầu tiên của faiss. 2 dòng dưới là workaround chuẩn.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss
faiss.omp_set_num_threads(1)  # tránh xung đột OpenMP kép trên macOS

try:
    # Local run: nạp secrets từ file .env (trên Colab/Kaggle dùng Secrets như hướng dẫn)
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")
# Fallback khi không có Groq key: endpoint OpenAI-compatible (vd OpenRouter)
EXTRACT_MODEL = get_secret("EXTRACT_MODEL", "")
OPENAI_BASE_URL = get_secret("OPENAI_BASE_URL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

# Đường dẫn: ưu tiên bản local trong repo; fallback /content cho Colab
DATA_PATH = ("data/hackernoon_subset.csv"
             if Path("data/hackernoon_subset.csv").exists()
             else "/content/hackernoon_subset.csv")
GOLDEN_DETAILED_PATH = "data/graphrag_golden_50_first5000_detailed.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Scale guard cho lab 2h
LAB_RAW_ROWS = 10_000        # Golden 50 xây từ ~5000 bài đầu; 10k dòng đầu CSV phủ đủ 51 bài evidence
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40
NEAR_DEDUP_THRESHOLD = 0.92  # cosine threshold cho bonus near-dedup (embedding + ANN)
print(f"DATA_PATH={DATA_PATH} · OUTPUT_DIR={OUTPUT_DIR.resolve()}")


DATA_PATH=/content/hackernoon_subset.csv · OUTPUT_DIR=/content/lab19/outputs


import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if OPENAI_BASE_URL:
    os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL
print("✅ Đã export secrets vào env cho OpenAI SDK")

In [29]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if OPENAI_BASE_URL:
    os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL
print("✅ Đã export secrets vào env cho OpenAI SDK")

✅ Đã export secrets vào env cho OpenAI SDK


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [30]:
#@title 1.3 — Stream dataset -> CSV (tự bỏ qua nếu đã có file local)
import csv
import os
from pathlib import Path
from tqdm.auto import tqdm

OUTPUT_CSV = DATA_PATH

if Path(OUTPUT_CSV).exists():
    _size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(f"✅ Dataset đã có sẵn: {OUTPUT_CSV} ({_size_mb:.1f} MB) — bỏ qua bước streaming.")
else:
    from datasets import load_dataset

    LIMIT_MB = 300           # hard-stop dung lượng (scale guard của lab)
    iterator = None
    transform = None
    headers = None

    if HF_TOKEN:
        # Thử bản gốc (gated) trước — cần token ĐÃ được cấp quyền truy cập dataset.
        try:
            DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
            dataset = load_dataset(DATASET_NAME, split="train", streaming=True, token=HF_TOKEN)
            iterator = iter(dataset)
            first_row = next(iterator)  # ép request đầu để phát hiện lỗi gated ngay tại đây
            headers = list(first_row.keys())
            LIMIT_ROWS = 1_000_000
            print(f"Dùng bản gốc: {DATASET_NAME}")
        except Exception as e:
            iterator = None
            print(f"⚠️ Không truy cập được bản gốc HackerNoon (token chưa được cấp quyền gated?)")
            print(f"   Chi tiết: {str(e)[:150]}")
            print("   -> Fallback sang mirror công khai.")

    if iterator is None:
        # Mirror công khai của cùng dữ liệu HackerNoon — không cần token.
        # Schema mirror KHÔNG có cột `text` -> tự ghép text = title + ". " + description
        # (đúng format bản CSV giảng viên cung cấp); bỏ cột `embedding` rất nặng.
        # Pipeline chỉ dùng LAB_RAW_ROWS=10k dòng đầu nên chỉ cần stream ~12k dòng.
        DATASET_NAME = "MongoDB/tech-news-embeddings"
        LIMIT_ROWS = 12_000
        FIELDS = ["companyName", "title", "description", "published_at", "url", "text"]
        def transform(row):
            t = str(row.get("title") or "").strip()
            d = str(row.get("description") or "").strip()
            return {
                "companyName": row.get("companyName"),
                "title": t,
                "description": d,
                "published_at": row.get("published_at"),
                "url": row.get("url"),
                "text": f"{t}. {d}".strip(". "),
            }
        print(f"Dùng mirror công khai: {DATASET_NAME}")
        dataset = load_dataset(DATASET_NAME, split="train", streaming=True)
        iterator = iter(dataset)
        first_row = transform(next(iterator))
        headers = FIELDS

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")
    rows_written = 0
    file_size_mb = 0.0
    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        with tqdm(total=LIMIT_ROWS, desc="Đang tải (rows)", unit="row") as pbar:
            pbar.update(1)
            for row in iterator:
                writer.writerow(transform(row) if transform else row)
                rows_written += 1
                pbar.update(1)

                if rows_written % 500 == 0:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    if file_size_mb >= LIMIT_MB:
                        print(f"\n[DỪNG] Đạt giới hạn dung lượng: {file_size_mb:.2f} MB ({rows_written:,} dòng)")
                        break
                if rows_written >= LIMIT_ROWS:
                    print(f"\n[DỪNG] Đạt giới hạn số dòng: {rows_written:,}")
                    break
        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)} — {rows_written:,} rows, {final_size_mb:.2f} MB")


✅ Dataset đã có sẵn: /content/hackernoon_subset.csv (7.9 MB) — bỏ qua bước streaming.


In [31]:
#@title 1.4 — Neo4j connection + schema (tự dò user/db; fallback cài Neo4j trong runtime)
driver = None

def _try_driver(uri, user, pw):
    d = GraphDatabase.driver(uri, auth=(user, pw))
    d.verify_connectivity()
    return d

def bootstrap_local_neo4j(password="lab19graphrag"):
    """Cài & khởi động Neo4j 5 ngay trong runtime (Colab/Linux) khi không có AuraDB.
    Dữ liệu mất khi runtime reset — chấp nhận được cho lab 2h."""
    import platform, shutil as _shutil, subprocess, urllib.request
    if platform.system() != "Linux":
        raise RuntimeError("Fallback tự cài Neo4j chỉ hỗ trợ Linux runtime (Colab). "
                           "Hãy cấu hình NEO4J_URI/NEO4J_PASSWORD trỏ tới server có sẵn.")
    # Java 17+ (yêu cầu của Neo4j 5)
    need_java = True
    if _shutil.which("java"):
        try:
            out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
            major = int(out.split('"')[1].split(".")[0])
            need_java = major < 17
        except Exception:
            pass
    if need_java:
        print("Cài OpenJDK 17...")
        subprocess.run(["apt-get", "-qq", "update"], check=False, capture_output=True)
        subprocess.check_call(["apt-get", "-qq", "install", "-y", "openjdk-17-jre-headless"],
                              stdout=subprocess.DEVNULL)

    VER = "5.26.0"
    home = Path("/content/neo4j-server")
    if not home.exists():
        tar = f"neo4j-community-{VER}-unix.tar.gz"
        print(f"Tải Neo4j {VER} (~170MB)...")
        urllib.request.urlretrieve(f"https://dist.neo4j.org/{tar}", f"/content/{tar}")
        subprocess.check_call(["tar", "-xzf", f"/content/{tar}", "-C", "/content"])
        Path(f"/content/neo4j-community-{VER}").rename(home)
        subprocess.check_call(
            [str(home / "bin" / "neo4j-admin"), "dbms", "set-initial-password", password],
            stdout=subprocess.DEVNULL,
        )
    subprocess.Popen([str(home / "bin" / "neo4j"), "console"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("Chờ Neo4j khởi động", end="")
    for _ in range(60):
        try:
            d = _try_driver("bolt://localhost:7687", "neo4j", password)
            print(" ✓")
            return d, "bolt://localhost:7687", "neo4j", password, "neo4j"
        except Exception:
            print(".", end="")
            time.sleep(3)
    raise RuntimeError("Neo4j local không khởi động được trong 180s.")

def connect_neo4j():
    global driver, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE
    if NEO4J_URI and NEO4J_PASSWORD:
        # File credentials của Aura có nơi ghi username/database = instance-id,
        # có nơi = 'neo4j' -> tự dò cả hai thay vì bắt người dùng đoán.
        users = [NEO4J_USER] + ([u for u in ["neo4j"] if u != NEO4J_USER])
        for u in users:
            try:
                driver = _try_driver(NEO4J_URI, u, NEO4J_PASSWORD)
                NEO4J_USER = u
                break
            except Exception as e:
                print(f"⚠️ {NEO4J_URI} user='{u}': {str(e)[:110]}")
        if driver is not None:
            for db in [NEO4J_DATABASE] + ([d for d in ["neo4j"] if d != NEO4J_DATABASE]):
                try:
                    with driver.session(database=db) as s:
                        s.run("RETURN 1").single()
                    NEO4J_DATABASE = db
                    print(f"✅ Neo4j connected: {NEO4J_URI} (user={NEO4J_USER}, db={NEO4J_DATABASE})")
                    return
                except Exception as e:
                    print(f"⚠️ database='{db}': {str(e)[:110]}")
            driver.close()
            driver = None
    print("→ Không nối được AuraDB/server cấu hình sẵn — fallback: cài Neo4j trong runtime.")
    driver, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE = bootstrap_local_neo4j()
    print(f"✅ Neo4j (local runtime) connected: {NEO4J_URI}")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

def reset_graph(batch=10000):
    """Xóa toàn bộ graph cũ để notebook idempotent khi Restart & Run All."""
    total = 0
    while True:
        deleted = run_cypher(
            "MATCH (n) WITH n LIMIT $batch DETACH DELETE n RETURN count(*) AS c",
            batch=batch,
        )[0]["c"]
        total += deleted
        if deleted == 0:
            break
    print(f"🧹 Graph reset: đã xóa {total:,} node cũ.")

connect_neo4j()
setup_graph_schema()
reset_graph()


⚠️  neo4j+s://9c549d06.databases.neo4j.io user='9c549d06': Failed to DNS resolve address 9c549d06.databases.neo4j.io:7687: [Errno -2] Name or service not known
⚠️  neo4j+s://9c549d06.databases.neo4j.io user='neo4j': Failed to DNS resolve address 9c549d06.databases.neo4j.io:7687: [Errno -2] Name or service not known
→ Không nối được AuraDB/server cấu hình sẵn — fallback: cài Neo4j trong runtime.
Chờ Neo4j khởi động ✓
✅ Neo4j (local runtime) connected: bolt://localhost:7687
✅ Schema ready.
🧹 Graph reset: đã xóa 0 node cũ.


In [32]:
#@title 1.5 — Loader + exact dedup + chunking (ưu tiên giữ bài evidence của Golden Dataset)
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path, nrows=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, nrows=nrows)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def load_golden_evidence_keys(path=GOLDEN_DETAILED_PATH):
    """Trích (published_at, title) của các bài evidence từ Golden detailed CSV.

    Golden 50 được sinh trên bản "first 5000" đã qua tiền xử lý riêng, nên row-id
    trong file golden KHÔNG khớp trực tiếp với CSV gốc. Khớp theo nội dung
    (ngày đăng + tiêu đề) bền vững hơn — đã xác minh đủ 51/51 bài trong 10k dòng đầu.
    """
    if not Path(path).exists():
        return set()
    det = pd.read_csv(path)
    keys = set()
    for ev in det.get("reference_evidence", pd.Series(dtype=str)).fillna(""):
        for m in re.finditer(r"row \d+ \(([^)]+)\): (.+?)(?= \| row |$)", str(ev)):
            keys.add((norm_space(m.group(1)), norm_space(m.group(2))))
    return keys

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""
    df["published_at_raw"] = raw[date_col].astype(str).map(norm_space) if date_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    # article_id đặt theo vị trí dòng gốc trong CSV để trace ngược dataset
    df["source_row"] = raw.index
    df["article_id"] = ["art_" + f"{int(i):06d}" for i in df["source_row"]]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    # Đánh dấu bài evidence của Golden Dataset để không bị loại khi cap số bài
    ev_keys = load_golden_evidence_keys()
    df["is_evidence"] = [
        (d, t) in ev_keys
        for d, t in zip(df["published_at_raw"], df["title"])
    ]
    n_ev = int(df["is_evidence"].sum())
    print(f"Bài evidence của Golden Dataset có mặt: {n_ev}/{len(ev_keys)}")

    # Cap số bài: giữ toàn bộ evidence, phần còn lại theo thứ tự xuất hiện
    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        ev = df[df["is_evidence"]]
        rest = df[~df["is_evidence"]].head(max(0, LAB_MAX_ARTICLES - len(ev)))
        df = (
            pd.concat([ev, rest])
            .sort_values("source_row")
            .reset_index(drop=True)
        )
    return df.drop(columns=["published_at_raw"]).reset_index(drop=True)

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
                "is_evidence": bool(r.is_evidence),
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH, nrows=LAB_RAW_ROWS)
news_df = standardize_news(raw_df)
print(f"Articles sau chuẩn hóa & cap: {len(news_df):,}")
display(news_df.head(3))


Exact dedup: 9,977 -> 3,751
Bài evidence của Golden Dataset có mặt: 52/51
Articles sau chuẩn hóa & cap: 1,500


,text,title,published_date,source_row,article_id,is_evidence
0,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications. (Nasdaq: ON) a leader in in...,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,0,art_000000,False
1,Adobe student receives national Information and Technology award. ELKO — An eighth grader at Adobe Middle School is ...,Adobe student receives national Information and Technology award,2023-05-02,1,art_000001,False
2,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery. To deliver 21st-century gove...,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,2,art_000002,False


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [33]:
#@title 1.5b — Bonus Near-Dedup: embedding + FAISS ANN (KHÔNG dùng pairwise O(N²))
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

def near_dedup(df, threshold=NEAR_DEDUP_THRESHOLD, top_k=5):
    """Bắt bài repost/near-duplicate mà exact SHA-1 hash bỏ sót.

    Thiết kế (Challenge A):
    - Embedding MiniLM + FAISS ANN top-k -> O(N·k), không pairwise O(N²).
    - threshold cosine 0.92: đủ cao để tránh false positive giữa các bài
      cùng chủ đề nhưng khác sự kiện.
    - Guard quan trọng: KHÔNG drop bài evidence của Golden Dataset — các câu
      cross-doc cần nhiều bài đưa tin về cùng một sự kiện (vd 2 bản tin
      Aeris–Ericsson tháng 12/2022 và 01/2023), gộp chúng sẽ phá tín hiệu.
    - Mọi quyết định đều ghi vào audit table để hậu kiểm.
    """
    df = df.reset_index(drop=True)
    texts = (df["title"].fillna("") + ". " + df["text"].str.slice(0, 400)).tolist()
    vecs = get_embedder().encode(
        texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True
    ).astype("float32")
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    sims, nbrs = index.search(vecs, min(top_k, len(df)))

    drop, audit = set(), []
    for i in range(len(df)):
        for score, j in zip(sims[i], nbrs[i]):
            j = int(j)
            if j <= i or float(score) < threshold:
                continue
            li, lj = df.iloc[i], df.iloc[j]
            if li.is_evidence or lj.is_evidence:
                decision = "KEEP_EVIDENCE"
            elif j in drop:
                decision = "ALREADY_DROPPED"
            else:
                decision = "DROP_NEAR_DUP"
                drop.add(j)
            audit.append({
                "left_id": li.article_id, "right_id": lj.article_id,
                "left_title": str(li.title)[:80], "right_title": str(lj.title)[:80],
                "similarity": round(float(score), 4), "decision": decision,
            })
    keep = df[~df.index.isin(drop)].reset_index(drop=True)
    return keep, pd.DataFrame(audit)

news_df, near_dedup_audit_df = near_dedup(news_df)
print(f"Near-dedup: còn {len(news_df):,} bài · audit {len(near_dedup_audit_df)} cặp")
near_dedup_audit_df.to_csv(OUTPUT_DIR / "near_dedup_audit.csv", index=False)
if len(near_dedup_audit_df):
    display(near_dedup_audit_df.sort_values("similarity", ascending=False).head(10))

chunks_df = build_chunks(news_df)
print(f"Chunks: {len(chunks_df):,} (evidence: {int(chunks_df.is_evidence.sum())})")
display(chunks_df.head())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Near-dedup: còn 1,447 bài · audit 69 cặp


,left_id,right_id,left_title,right_title,similarity,decision
19,art_000125,art_001439,Sensormatic Solutions by Johnson Controls shares 2022 U.S. Super Saturday shoppe,Sensormatic Solutions by Johnson Controls shares 2022 U.S. Super Saturday shoppe,0.9994,DROP_NEAR_DUP
22,art_000204,art_000688,Technology Services (CITE),Technology Services (CITE),0.9965,DROP_NEAR_DUP
54,art_001333,art_001337,US announces criminal cases involving flow of technology information to Russia C,US announces criminal cases involving flow of technology information to Russia C,0.9965,DROP_NEAR_DUP
9,art_000043,art_002987,Samsung Showcases Groundbreaking Logic Innovations at System LSI Tech Day 2023,Samsung Showcases Groundbreaking Logic Innovations at System LSI Tech Day 2023,0.9955,KEEP_EVIDENCE
51,art_001068,art_001260,Information Services Group Inc. (NASDAQ:III) is a favorite amongst institutional,Information Services Group Inc. (NASDAQ:III) is a favorite amongst institutional,0.9955,DROP_NEAR_DUP
62,art_001811,art_001950,SIOS Technology Announces Cloud Availability Symposium 2023: Disaster Recovery M,SIOS Technology Announces Cloud Availability Symposium 2023: Disaster Recovery M,0.9949,DROP_NEAR_DUP
44,art_000829,art_001012,Russian tech giant Yandex says code leaked in cybersecurity incident,Russian tech giant Yandex says code leaked in cybersecurity incident,0.9946,DROP_NEAR_DUP
14,art_000073,art_000268,The Institution of Engineering and Technology (IET) successfully concludes IET F,The Institution of Engineering and Technology (IET) successfully concludes IET F,0.9943,DROP_NEAR_DUP
45,art_000895,art_003072,Truescope Acquires US Firm Universal Information Services,Truescope Acquires US Firm Universal Information Services,0.9936,ALREADY_DROPPED
52,art_001157,art_001346,CereCore® expands healthcare technology services into the UK,CereCore expands healthcare technology services into the UK,0.9923,DROP_NEAR_DUP


Chunking:   0%|          | 0/1447 [00:00<?, ?it/s]

Chunks: 1,447 (evidence: 52)


,chunk_id,article_id,title,published_date,text,is_evidence
0,art_000000::c0000,art_000000,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications. (Nasdaq: ON) a leader in in...,False
1,art_000001::c0000,art_000001,Adobe student receives national Information and Technology award,2023-05-02,Adobe student receives national Information and Technology award. ELKO — An eighth grader at Adobe Middle School is ...,False
2,art_000002::c0000,art_000002,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery. To deliver 21st-century gove...,False
3,art_000003::c0000,art_000003,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity. In February GreenPages acqu...,False
4,art_000004::c0000,art_000004,Synex Renewable Energy Corporation (Formerly Synex International Inc.) Third Quarter of Fiscal 2023,2023-05-15,Synex Renewable Energy Corporation (Formerly Synex International Inc.) Third Quarter of Fiscal 2023. The conference ...,False


In [34]:
#@title 1.6 — LLM wrapper có retry + JSON parsing (Groq hoặc endpoint OpenAI-compatible)
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# Fallback: không có Groq key -> endpoint OpenAI-compatible (vd OpenRouter).
# SDK OpenAI tự đọc OPENAI_API_KEY / OPENAI_BASE_URL từ biến môi trường.
openai_gen_client = None
if groq_client is None:
    if not (OPENAI_API_KEY and EXTRACT_MODEL):
        raise RuntimeError("Cần GROQ_API_KEY, hoặc OPENAI_API_KEY + EXTRACT_MODEL (OpenAI-compatible).")
    from openai import OpenAI
    openai_gen_client = OpenAI()

GEN_MODEL = GROQ_MODEL if groq_client else EXTRACT_MODEL
print(f"Generator/Extractor: {'Groq' if groq_client else 'OpenAI-compatible'} · model={GEN_MODEL}")

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    """Giữ tên hàm theo lab guide; tự route sang client khả dụng (Groq/OpenAI-compatible)."""
    client = groq_client or openai_gen_client
    model = model or GEN_MODEL
    if not model:
        raise RuntimeError("Thiếu model cho generator (GROQ_MODEL hoặc EXTRACT_MODEL).")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            try:
                resp = client.chat.completions.create(**kwargs)
            except Exception as e:
                # Một số provider không hỗ trợ response_format -> thử lại không ép JSON
                if json_mode and "response_format" in str(e).lower():
                    kwargs.pop("response_format", None)
                    resp = client.chat.completions.create(**kwargs)
                else:
                    raise

            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage


Generator/Extractor: OpenAI-compatible · model=openai/gpt-4o-mini


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [35]:
#@title 1.7 — Coreference resolution theo batch (concurrent + cache)
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5, max_workers=4):
    starts = list(range(0, len(chunks_subset), batch_size))
    results = [None] * len(starts)

    def work(pos):
        batch = chunks_subset.iloc[starts[pos]:starts[pos] + batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        return pos, df

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(work, p) for p in range(len(starts))]
        for f in tqdm(as_completed(futures), total=len(futures), desc="Coref"):
            pos, df = f.result()
            results[pos] = df
    return pd.concat(results, ignore_index=True)

def select_extraction_chunks(chunks_df, cap=EXTRACTION_MAX_CHUNKS):
    """Chunk evidence của Golden vào trước để graph chắc chắn chứa dữ kiện cần
    trả lời; phần còn lại lấy theo thứ tự corpus cho tới cap (scale guard)."""
    ev = chunks_df[chunks_df["is_evidence"]]
    rest = chunks_df[~chunks_df["is_evidence"]].head(max(0, cap - len(ev)))
    return pd.concat([ev, rest]).drop_duplicates("chunk_id").head(cap).reset_index(drop=True)

COREF_CACHE = OUTPUT_DIR / "coref_cache.csv"
extraction_source = select_extraction_chunks(chunks_df)
print(f"Extraction source: {len(extraction_source)} chunks (evidence: {int(extraction_source.is_evidence.sum())})")

if COREF_CACHE.exists():
    coref_df = pd.read_csv(COREF_CACHE)
    coref_df["unresolved_mentions"] = coref_df["unresolved_mentions"].fillna("[]").map(json.loads)
    print(f"♻️ Nạp coref cache: {len(coref_df)} chunks (xóa {COREF_CACHE} nếu muốn chạy lại)")
else:
    coref_df = run_coref(extraction_source)
    _save = coref_df.copy()
    _save["unresolved_mentions"] = _save["unresolved_mentions"].map(json.dumps)
    _save.to_csv(COREF_CACHE, index=False)

extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")
extraction_source["resolved_text"] = extraction_source["resolved_text"].fillna(extraction_source["text"])
extraction_source["unresolved_mentions"] = extraction_source["unresolved_mentions"].apply(
    lambda x: x if isinstance(x, list) else []
)

_changed = extraction_source[
    extraction_source["resolved_text"].str.strip() != extraction_source["text"].str.strip()
]
_n_unres = int(extraction_source["unresolved_mentions"].map(len).sum())
print(f"Coref: sửa {len(_changed)}/{len(extraction_source)} chunks · tổng unresolved mentions: {_n_unres}")
if len(_changed):
    _ex = _changed.iloc[0]
    print(f"--- Spot-check (chunk {_ex.chunk_id}) ---")
    print("GỐC     :", _ex.text[:400])
    print("RESOLVED:", _ex.resolved_text[:400])


Extraction source: 400 chunks (evidence: 52)


Coref:   0%|          | 0/80 [00:00<?, ?it/s]

Coref: sửa 264/400 chunks · tổng unresolved mentions: 17
--- Spot-check (chunk art_000033::c0000) ---
GỐC     : Aeris to Acquire IoT Business from Ericsson. Aeris Communications and Ericsson are joining together to create a leader in the fast-growing IoT industry Ericsson''s IoT Accelerator and Connected Vehicle Cloud businesses and related assets to be
RESOLVED: Aeris to Acquire IoT Business from Ericsson. Aeris Communications and Ericsson are joining together to create a leader in the fast-growing IoT industry. Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses and related assets are to be acquired.


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [36]:
#@title 2.1 — NER + RE extraction (concurrent + cache)
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4, max_workers=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    starts = list(range(0, len(source_df), batch_size))
    results = [None] * len(starts)
    errors = []

    def work(pos):
        batch = source_df.iloc[starts[pos]:starts[pos] + batch_size]
        return extract_batch(batch)[0]

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(work, p): p for p in range(len(starts))}
        for f in tqdm(as_completed(futures), total=len(futures), desc="NER+RE"):
            p = futures[f]
            try:
                results[p] = f.result()
            except Exception as e:
                errors.append({"start": starts[p], "error": str(e)})

    triples = []
    for obj in results:
        if not obj:
            continue
        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    return pd.DataFrame(triples), pd.DataFrame(errors)

TRIPLES_CACHE = OUTPUT_DIR / "raw_triples_cache.csv"
if TRIPLES_CACHE.exists():
    raw_triples_df = pd.read_csv(TRIPLES_CACHE, keep_default_na=False)
    raw_triples_df["confidence"] = pd.to_numeric(raw_triples_df["confidence"], errors="coerce").fillna(0.0)
    extraction_errors_df = pd.DataFrame()
    print(f"♻️ Nạp triples cache: {len(raw_triples_df)} (xóa {TRIPLES_CACHE} nếu muốn chạy lại)")
else:
    raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
    raw_triples_df.to_csv(TRIPLES_CACHE, index=False)

print(f"Triples hợp lệ: {len(raw_triples_df)} · batch lỗi: {len(extraction_errors_df)}")
display(raw_triples_df.head())


NER+RE:   0%|          | 0/100 [00:00<?, ?it/s]

Triples hợp lệ: 265 · batch lỗi: 0


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Aeris Communications,Company,ACQUIRED,IoT Business from Ericsson,Technology,art_000033::c0000,2022-12-07,Aeris to Acquire IoT Business from Ericsson.,1.0
1,Aeris Communications,Company,PARTNERED_WITH,Ericsson,Company,art_000033::c0000,2022-12-07,Aeris Communications and Ericsson are joining together.,1.0
2,Samsung Electronics Co. Ltd.,Company,DEVELOPED,advanced semiconductor technology,Technology,art_000043::c0000,2023-10-05,"Samsung Electronics Co. Ltd., a world leader in advanced semiconductor technology.",1.0
3,Samsung,Company,PARTNERED_WITH,Aqara,Company,art_000078::c0000,2023-10-05,Samsung and Aqara Partners to Demonstrate Presence Sensor FP2.,1.0
4,Aqara,Company,DEVELOPED,FP2 Presence Sensor,Technology,art_000078::c0000,2023-10-05,The FP2 Presence Sensor is Aqara's latest occupancy sensor.,1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [37]:
#@title 2.2 — Entity resolution (Vector ANN + Lexical Guard + Union-Find)
CORP_SUFFIXES = {
    "inc", "incorporated", "corp", "corporation", "ltd", "limited", "llc", "plc",
    "co", "company", "platforms", "technologies", "technology", "solutions",
    "systems", "group", "holdings", "labs", "international", "communications",
}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b, typ=None):
    """Lexical guard chống false merge (Challenge B):
    - Person: họ (token cuối) phải trùng tuyệt đối; tên phải là prefix/viết tắt
      của nhau -> chặn 'Sam Altman' vs 'Steve Altman' dù vector similarity cao.
    - Company/Technology: nếu tên này là token-subset thực sự của tên kia và
      phần thừa không phải hậu tố doanh nghiệp (Inc/Corp/Platforms...) thì nghi
      là product/sub-brand -> chặn 'Apple' vs 'Apple Watch', 'Microsoft' vs
      'Microsoft Teams' (nhưng 'Meta' vs 'Meta Platforms' vẫn merge nhờ strip suffix).
    """
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    if typ == "Person":
        ta, tb = na.split(), nb.split()
        if not ta or not tb or ta[-1] != tb[-1]:
            return False
        fa, fb = ta[0], tb[0]
        return fa.startswith(fb) or fb.startswith(fa)
    sa, sb = set(na.split()), set(nb.split())
    if sa != sb and (sa <= sb or sb <= sa):
        # Ngoại lệ: token thừa là chữ viết tắt (initials) của phần còn lại
        # -> 'Amazon Web Services' vs 'Amazon Web Services (AWS)' vẫn được merge
        big, small = (na, nb) if sb <= sa else (nb, na)
        extras = [t for t in big.split() if t not in set(small.split())]
        initials = "".join(w[0] for w in small.split())
        if len(extras) == 1 and extras[0] == initials:
            return True
        return False
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

AUDIT_LOG_SIM = 0.80  # log mọi cặp candidate >= 0.80 vào audit (kể cả không merge)

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                score = float(score)
                if j < 0 or i >= j or score < AUDIT_LOG_SIM:
                    continue
                if score < threshold:
                    # Dưới ngưỡng merge nhưng vẫn ghi audit để hậu kiểm near-miss
                    audit.append({
                        "type": typ, "left": names[i], "right": names[j],
                        "similarity": score, "decision": "REJECT_THRESHOLD"
                    })
                    continue
                ok = merge_guard(names[i], names[j], typ)
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": score,
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
entity_resolution_audit_df.to_csv(OUTPUT_DIR / "entity_resolution_audit.csv", index=False)
print(f"Audit rows: {len(entity_resolution_audit_df)} · triples sau canonicalize: {len(triples_df)}")
print(entity_resolution_audit_df["decision"].value_counts().to_dict() if len(entity_resolution_audit_df) else "audit trống")
display(entity_resolution_audit_df.head(20))


Audit rows: 14 · triples sau canonicalize: 264
{'REJECT_THRESHOLD': 11, 'MERGE_VECTOR': 3}


,type,left,right,similarity,decision
0,Company,L T Technology Services,L&T Technology Services Limited,0.894427,REJECT_THRESHOLD
1,Company,L T Technology Services,L&T Technology Services Ltd.,0.842785,REJECT_THRESHOLD
2,Company,NVIDIA,Nvidia Corp.,0.827687,REJECT_THRESHOLD
3,Company,Google Cloud,Google Cloud Platform,0.892152,REJECT_THRESHOLD
4,Company,Amazon Web Services,Amazon Web Services (AWS),0.928739,MERGE_VECTOR
5,Company,Fidelity National Information Services,Fidelity National Information Services Inc.,0.924524,MERGE_VECTOR
6,Company,SIOS Technology,SIOS Technology Corp.,0.842801,REJECT_THRESHOLD
7,Company,Synechron Inc.,Synechron,0.827904,REJECT_THRESHOLD
8,Company,L&T Technology Services Ltd.,L&T Technology Services Limited,0.890695,REJECT_THRESHOLD
9,Technology,generative AI,generative AI capabilities,0.856484,REJECT_THRESHOLD


In [38]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
print(f"Nodes: {len(nodes_df)} · Edges chuẩn bị insert: {len(triples_df)}")
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)
print("✅ Bulk insert xong (UNWIND, batch 1000 — không insert từng row).")


Nodes: 399 · Edges chuẩn bị insert: 264
✅ Bulk insert xong (UNWIND, batch 1000 — không insert từng row).


In [39]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()
top_degree_df.to_csv(OUTPUT_DIR / "top_degree_nodes.csv", index=False)


{'nodes': 399, 'edges': 264, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,fb0f4df56fab164ec48722f0,Microsoft,Company,17
1,cc9c6ee3857729e221d3f6de,ServiceNow,Company,9
2,773eeb9b7cc008bff365fcdd,OpenAI,Company,7
3,7b29988cfc0dac3059f47a0e,L T Technology Services,Company,5
4,110cfb16be66531da841ce26,Thales,Company,5
5,24efe6cedc09a12ec8c3eff2,Synopsys,Company,5
6,f4adaa883e92217c80dab7fb,ChatGPT,Technology,4
7,8aaf4c8f85ca26a0418b8601,Dell,Company,4
8,fd90caf986a66450a7dd026b,Intel,Company,4
9,909fcd9c188c8c2429afa468,Google Cloud,Company,4


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [40]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Flat vectors: 1447


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [41]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)
print(f"Entity matcher: {len(entity_match_store)} nodes đã index cho fuzzy fallback.")


Entity matcher: 399 nodes đã index cho fuzzy fallback.


In [42]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [43]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GEN_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    # latency đo END-TO-END (retrieval + generation) để so sánh công bằng
    t0 = time.perf_counter()
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out["latency_s"] = time.perf_counter() - t0
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    # latency gồm cả seed extraction (1 LLM call) + BFS Neo4j + generation
    t0 = time.perf_counter()
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out["latency_s"] = time.perf_counter() - t0
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# --- Demo nhanh trên sự kiện có thật trong corpus (Aeris mua mảng IoT của Ericsson) ---
_demo_q = "Which Ericsson businesses did Aeris agree to acquire, and when were the deals reported?"
_flat_demo = answer_flat_rag(_demo_q)
_graph_demo = answer_graph_rag(_demo_q)
print("=== FLAT RAG ===")
print(_flat_demo["answer"][:900])
print()
print("=== HYBRID GRAPHRAG ===")
print(_graph_demo["answer"][:900])
print()
_diag = _graph_demo["graph_debug"]["diagnostics"]
print("Seeds matched:", [s["name"] for s in _diag.get("matched_seeds", [])])
print("Diagnostics:", {k: v for k, v in _diag.items() if k != "matched_seeds"})
if len(_graph_demo["graph_debug"]["edges"]):
    display(_graph_demo["graph_debug"]["edges"].head(8))


=== FLAT RAG ===
Aeris agreed to acquire Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses and related assets. The deals were reported on December 7, 2022, and further details were highlighted in reports on January 10, 2023, and January 18, 2023 [chunk_id=art_000033::c0000, chunk_id=art_002840::c0000, chunk_id=art_000836::c0000].

=== HYBRID GRAPHRAG ===
Aeris agreed to acquire Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses. The deals were reported on January 10, 2023, and finalized on January 18, 2023 [chunk_id=art_000836::c0000].

Seeds matched: ['Ericsson', 'Aeris']
Diagnostics: {'expanded_nodes': 5, 'collected_edges': 5, 'supernode_events': []}


,source_id,source_name,source_type,relation,target_id,target_name,target_type,source_chunk_id,published_date,evidence,neighbor_id
0,ce0b6a8e50a6a58526e4c85f,Aeris,Company,ACQUIRED,02f50203f5a967ac3ebb42be,Ericsson,Company,art_000836::c0000,2023-01-18,Aeris has acquired Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses,ce0b6a8e50a6a58526e4c85f
1,02f50203f5a967ac3ebb42be,Ericsson,Company,WORKED_AT,a66eb827ecadf31054864df3,IoT Accelerator and Connected Vehicle Cloud businesses,Technology,art_002840::c0000,2023-01-10,Highlights Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses.,a66eb827ecadf31054864df3
2,0ecbd5177fafb6f25a33707e,Aeris Communications,Company,PARTNERED_WITH,02f50203f5a967ac3ebb42be,Ericsson,Company,art_000033::c0000,2022-12-07,Aeris Communications and Ericsson are joining together.,0ecbd5177fafb6f25a33707e
3,ce0b6a8e50a6a58526e4c85f,Aeris,Company,ACQUIRED,a1f1d537338e100dc6354636,IoT Business from Ericsson,Technology,art_002840::c0000,2023-01-10,Aeris to acquire IoT business from Ericsson.,a1f1d537338e100dc6354636
4,0ecbd5177fafb6f25a33707e,Aeris Communications,Company,ACQUIRED,a1f1d537338e100dc6354636,IoT Business from Ericsson,Technology,art_000033::c0000,2022-12-07,Aeris to Acquire IoT Business from Ericsson.,a1f1d537338e100dc6354636


# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [44]:
#@title 4.1 — Golden Dataset (50 câu factoid/multi-hop/cross-doc, đã có reference answers)
GOLDEN_PATH = ("data/graphrag_golden_50_first5000.csv"
               if Path("data/graphrag_golden_50_first5000.csv").exists()
               else "/content/golden_dataset.csv")

# Starter 5 câu của lab guide — chỉ dùng làm fallback khi không có file golden đầy đủ
starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startups were founded by former Microsoft employees and later received investment from Google?",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Find a company invested in by a major technology company that also developed a named AI technology; identify both relations and dates.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
])

golden_df = pd.read_csv(GOLDEN_PATH) if Path(GOLDEN_PATH).exists() else starter_golden.copy()
print(f"Golden Dataset: {GOLDEN_PATH} — {len(golden_df)} câu · nhóm: {golden_df.group.value_counts().to_dict()}")
display(golden_df.head())

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

validate_golden(golden_df, require_answers=True)


Golden Dataset: data/graphrag_golden_50_first5000.csv — 50 câu · nhóm: {'multi-hop': 23, 'cross-doc': 22, 'factoid': 5}


,id,group,question,reference_answer,reference_evidence
0,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...",row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
1,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
2,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.",row 935 (2023-01-18 22:37:00): A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular...
3,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
4,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 935 (2023-01-18 22:37:00): A Leap in...


✅ Golden Dataset valid.


In [45]:
#@title 4.2 — LLM-as-a-Judge (model khác generator để giảm self-preference bias)
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

if JUDGE_PROVIDER not in {"openai", "groq"}:
    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_json(system, user, max_retries=3):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")
    last = None
    for attempt in range(max_retries):
        try:
            if JUDGE_PROVIDER == "groq":
                return groq_json(system, user, model=JUDGE_MODEL)[0]
            # openai / OpenAI-compatible (SDK tự đọc OPENAI_BASE_URL từ env, vd OpenRouter)
            if not OPENAI_API_KEY:
                raise RuntimeError("Thiếu OPENAI_API_KEY.")
            from openai import OpenAI
            client = OpenAI(api_key=OPENAI_API_KEY)
            kwargs = dict(
                model=JUDGE_MODEL,
                messages=[{"role":"system","content":system},
                          {"role":"user","content":user}],
                temperature=0.0,
            )
            try:
                resp = client.chat.completions.create(
                    **kwargs, response_format={"type":"json_object"}
                )
            except Exception as e:
                if "response_format" in str(e).lower():
                    resp = client.chat.completions.create(**kwargs)
                else:
                    raise
            return parse_json_object(resp.choices[0].message.content)
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(15, 2**attempt + random.random()))
    raise RuntimeError(last)

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(float(obj.get(k, 1)))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out


In [46]:
#@title 4.3 — Evaluation runner + checkpoint (resume được nếu bị ngắt giữa chừng)
CHECKPOINT = str(OUTPUT_DIR / "graphrag_eval_checkpoint.csv")

def run_evaluation(golden_df, checkpoint=CHECKPOINT):
    done = pd.read_csv(checkpoint) if Path(checkpoint).exists() else pd.DataFrame()
    done_ids = set(done["id"]) if len(done) else set()
    rows = done.to_dict("records")
    todo = golden_df[~golden_df["id"].isin(done_ids)]
    if done_ids:
        print(f"♻️ Resume từ checkpoint: {len(done_ids)} câu đã có, còn {len(todo)} câu")

    for q in tqdm(todo.itertuples(index=False), total=len(todo), desc="Evaluation"):
        try:
            flat = answer_flat_rag(q.question)
            graph = answer_graph_rag(q.question)

            jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
            jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])
        except Exception as e:
            print(f"⚠️ {q.id} lỗi: {e} — bỏ qua (chạy lại cell để retry câu này).")
            continue

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(checkpoint, index=False)

    out = pd.DataFrame(rows)
    if len(out):
        order = [i for i in golden_df["id"] if i in set(out["id"])]
        out = out.set_index("id").loc[order].reset_index()
    return out

eval_results_df = run_evaluation(golden_df)
print(f"Đã đánh giá {len(eval_results_df)}/{len(golden_df)} câu.")
display(eval_results_df.head(10))


Evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

Đã đánh giá 50/50 câu.


,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...","Aeris Communications acquired Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, along with related ...","Aeris acquired Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, along with related assets, as part...",5,5,5,5,5,5,1.869717,3.836302,686,769,The candidate accurately identifies the Ericsson businesses transferred to Aeris (IoT Accelerator and Connected Vehi...,The candidate accurately identifies the Ericsson businesses transferred to Aeris (IoT Accelerator and Connected Vehi...,0
1,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,"The first two Aeris/Ericsson reports described a planned transfer of Ericsson's IoT business to Aeris, indicating th...","The first two Aeris/Ericsson reports describe a planned transfer of Ericsson's IoT business to Aeris. Specifically, ...",5,5,5,5,5,5,2.482374,3.902620,680,804,,The candidate accurately identifies that the first reports describe a planned transfer and correctly points to the J...,0
2,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.","The later connectivity report cited more than 100 million IoT devices, 9,000 enterprises, and 190 countries [chunk_i...","The later connectivity report cited more than 100 million IoT devices, 9,000 enterprises, and 190 countries [chunk_i...",5,5,5,5,5,5,1.555144,2.646886,620,700,"The candidate answer directly and accurately provides all the requested information (number of IoT devices, enterpri...","The candidate answer directly and accurately provides all the requested information (number of IoT devices, enterpri...",0
3,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,The two named Ericsson IoT businesses that recur across multiple reports of the Aeris transaction are the IoT Accele...,The two named Ericsson IoT businesses that recur across multiple reports of the Aeris transaction are the IoT Accele...,5,5,5,5,5,5,2.581813,3.265823,672,764,,,0
4,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,The path from Ericsson to the acquirer and then to the reported IoT reach is as follows:\n\n1. **Starting Point**: E...,The path from Ericsson to the acquirer and then to the reported IoT reach is as follows:\n\n1. **Acquirer**: Aeris a...,5,5,5,5,5,5,1.628253,4.030085,689,786,"The candidate accurately identifies Ericsson as the starting point, Aeris as the acquirer, and correctly states the ...","The candidate accurately identifies Aeris as the acquirer and correctly states the reported IoT reach, including the...",0
5,G5000-06,multi-hop,Trace ServiceNow's generative-AI product/partner evolution from May through July 2023: what partnership began in May..

In [47]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    groups = list(eval_df.groupby("group")) + [("TẤT CẢ", eval_df)]
    for group, g in groups:
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv(OUTPUT_DIR / "graphrag_eval_results.csv", index=False)
comparison_df.to_csv(OUTPUT_DIR / "graphrag_vs_flatrag_summary.csv", index=False)
print("✅ Đã xuất outputs/graphrag_eval_results.csv & outputs/graphrag_vs_flatrag_summary.csv")


,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,3.182,4.136,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
1,cross-doc,Faithfulness,3.773,4.409,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,3.545,4.136,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),2.291,3.751,Flat RAG thường rẻ/nhanh hơn.
4,cross-doc,Token usage,672.727,1321.455,Flat RAG thường rẻ/nhanh hơn.
5,factoid,Comprehensiveness,5.000,5.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,5.000,4.600,Hai phương pháp gần nhau.
8,factoid,Latency (s),1.518,2.540,Flat RAG thường rẻ/nhanh hơn.
9,factoid,Token usage,627.600,951.600,Flat RAG thường rẻ/nhanh hơn.


✅ Đã xuất outputs/graphrag_eval_results.csv & outputs/graphrag_vs_flatrag_summary.csv


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [48]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")
    else:
        print(f"ℹ️ Không có node degree > {SUPER_NODE_DEGREE} trong lab subset -> chạy thêm test mô phỏng bên dưới.")

def test_supernode_policy_simulated():
    """Lab subset (400 chunks) có thể chưa tạo ra node degree > 100.
    Mô phỏng: ép node bậc cao nhất qua policy cap 50 và kiểm tra bất biến:
    (1) số edge trả về <= SUPER_NODE_EDGE_CAP, (2) sort published_date DESC."""
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        return
    n = rows[0]
    edges = recent_edges(n["id"], SUPER_NODE_EDGE_CAP)
    dates = [e.get("published_date") or "" for e in edges]
    assert len(edges) <= SUPER_NODE_EDGE_CAP, "Cap 50 edge bị vi phạm"
    assert dates == sorted(dates, reverse=True), "Edge không sort theo published_date DESC"
    print(f"✅ Mô phỏng cap: node '{n['name']}' (degree={n['degree']}) -> fetched {len(edges)} <= {SUPER_NODE_EDGE_CAP}, ưu tiên mới nhất trước.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
test_supernode_policy_simulated()
show_resolution_audit(entity_resolution_audit_df)


{'id': 'fb0f4df56fab164ec48722f0', 'name': 'Microsoft', 'degree': 17} fetched= 17
ℹ️ Không có node degree > 100 trong lab subset -> chạy thêm test mô phỏng bên dưới.
✅ Mô phỏng cap: node 'Microsoft' (degree=17) -> fetched 17 <= 50, ưu tiên mới nhất trước.


,type,left,right,similarity,decision
10,Technology,AI service,AI services,0.967647,MERGE_VECTOR
4,Company,Amazon Web Services,Amazon Web Services (AWS),0.928739,MERGE_VECTOR
5,Company,Fidelity National Information Services,Fidelity National Information Services Inc.,0.924524,MERGE_VECTOR
0,Company,L T Technology Services,L&T Technology Services Limited,0.894427,REJECT_THRESHOLD
3,Company,Google Cloud,Google Cloud Platform,0.892152,REJECT_THRESHOLD
8,Company,L&T Technology Services Ltd.,L&T Technology Services Limited,0.890695,REJECT_THRESHOLD
9,Technology,generative AI,generative AI capabilities,0.856484,REJECT_THRESHOLD
6,Company,SIOS Technology,SIOS Technology Corp.,0.842801,REJECT_THRESHOLD
1,Company,L T Technology Services,L&T Technology Services Ltd.,0.842785,REJECT_THRESHOLD
12,Technology,AI cloud services,AI services,0.842450,REJECT_THRESHOLD


High-similarity rejected pairs:


,type,left,right,similarity,decision


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

In [49]:
#@title 5.3 — Submission self-check (theo Submission Checklist của README/RUBRIC)
_invalid_edges = run_cypher("""
MATCH ()-[r]->()
WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
RETURN count(r) AS n
""")[0]["n"]

checks = {
    "Neo4j connected & schema": driver is not None,
    "0 edge thiếu provenance": _invalid_edges == 0,
    "Entity audit >= 10 dòng": len(entity_resolution_audit_df) >= 10,
    "Golden đủ 3 nhóm câu hỏi": {"factoid", "multi-hop", "cross-doc"} <= set(golden_df.group.unique()),
    "Golden có reference answers": not golden_df.reference_answer.fillna("").str.strip().eq("").any(),
    "Eval chạy đủ số câu": len(eval_results_df) == len(golden_df),
    "outputs/graphrag_eval_results.csv": (OUTPUT_DIR / "graphrag_eval_results.csv").exists(),
    "outputs/graphrag_vs_flatrag_summary.csv": (OUTPUT_DIR / "graphrag_vs_flatrag_summary.csv").exists(),
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)
assert all(checks.values()), "Có mục chưa đạt — xem ❌ ở trên."
print("\n🎉 Tất cả kiểm tra bắt buộc đã đạt.")


✅ Neo4j connected & schema
✅ 0 edge thiếu provenance
✅ Entity audit >= 10 dòng
✅ Golden đủ 3 nhóm câu hỏi
✅ Golden có reference answers
✅ Eval chạy đủ số câu
✅ outputs/graphrag_eval_results.csv
✅ outputs/graphrag_vs_flatrag_summary.csv

🎉 Tất cả kiểm tra bắt buộc đã đạt.


In [50]:
#@title 5.4 — (Colab) Nén outputs/ để tải về máy và commit vào repo nộp bài
import shutil
_zip = shutil.make_archive("lab19_outputs", "zip", "outputs")
print(f"✅ Đã tạo {_zip}")
print("Trên Colab: mở Files panel (📁 bên trái) -> tải lab19_outputs.zip về,")
print("giải nén vào thư mục outputs/ của repo local rồi commit + push khi nộp bài.")


✅ Đã tạo /content/lab19/lab19_outputs.zip
Trên Colab: mở Files panel (📁 bên trái) -> tải lab19_outputs.zip về,
giải nén vào thư mục outputs/ của repo local rồi commit + push khi nộp bài.


# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [51]:
#@title Bonus — Global Search: NetworkX community detection + community reports
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

community_df = build_communities()
_sizes = community_df["community_id"].value_counts()
print(f"Communities: {_sizes.shape[0]} · top-5 kích thước: {_sizes.head(5).to_dict()}")

def summarize_community(cid, max_edges=40):
    edges = run_cypher("""
    MATCH (a:Entity {community_id:$cid})-[r]->(b:Entity {community_id:$cid})
    RETURN a.name AS s, type(r) AS rel, b.name AS t,
           r.published_date AS date
    ORDER BY coalesce(r.published_date,'') DESC LIMIT $k
    """, cid=int(cid), k=int(max_edges))
    if not edges:
        return None
    facts = "\n".join(f"{e['s']} -{e['rel']}-> {e['t']} ({e['date']})" for e in edges)
    obj, _ = groq_json(
        'Summarize this knowledge-graph community for global search. '
        'Return JSON {"title":"...","summary":"3-4 sentences"}.',
        f"EDGES:\n{facts}",
    )
    return {
        "community_id": int(cid),
        "n_members": int(_sizes[cid]),
        "title": norm_space(obj.get("title")),
        "summary": norm_space(obj.get("summary")),
    }

community_reports_df = pd.DataFrame(
    [r for r in (summarize_community(c) for c in _sizes.head(5).index) if r]
)
community_reports_df.to_csv(OUTPUT_DIR / "community_reports.csv", index=False)
display(community_reports_df)

# Global search demo: câu hỏi vĩ mô trả lời từ community reports (không cần seed entity)
_global_q = "What are the main clusters of tech-company activity in this news corpus, and what characterizes each cluster?"
_global_ctx = "\n\n".join(
    f"[Community {r.community_id} | {r.title} | {r.n_members} members] {r.summary}"
    for r in community_reports_df.itertuples(index=False)
)
print("=== GLOBAL SEARCH ANSWER ===")
print(generate_answer(_global_q, _global_ctx)["answer"])


Communities: 149 · top-5 kích thước: {0: 23, 1: 12, 2: 12, 3: 8, 4: 7}


,community_id,n_members,title,summary
0,0,23,Global Search Knowledge Graph Community,This knowledge graph highlights key partnerships and technological developments among major players in the cloud and...
1,1,12,Global Partnerships and AI Development in Tech,"The knowledge graph highlights significant partnerships and developments in the tech industry, particularly focusing..."
2,2,12,Knowledge Graph Community for Global Search,This knowledge graph highlights key partnerships and developments involving major tech companies and individuals. No...
3,3,8,Dell Technologies Innovations and Partnerships,Dell Technologies Inc. has been actively developing innovative solutions and forming strategic partnerships to enhan...
4,4,7,Recent Collaborations and Developments in Semiconductor Technology,The knowledge graph highlights key partnerships and technological advancements in the semiconductor industry. Notabl...


=== GLOBAL SEARCH ANSWER ===
The main clusters of tech-company activity in the news corpus are:

1. **Cloud and AI Sector**:
   - **Key Players**: Microsoft, Google Cloud, AWS, KPMG, Options Technology.
   - **Characteristics**: Focus on partnerships and technological advancements in cloud services and AI. Microsoft is developing services like Azure OpenAI and Copilot, while Google Cloud utilizes advanced AI models such as Code Llama and Llama 2 [chunk_id=0].

2. **Generative AI Development**:
   - **Key Players**: ServiceNow, Amazon, Deloitte, Accenture, NVIDIA.
   - **Characteristics**: Emphasis on strategic alliances and the advancement of generative AI capabilities across various sectors. Amazon is also developing AI services in collaboration with partners like Cohere [chunk_id=1].

3. **Tech Partnerships and Innovations**:
   - **Key Players**: OpenAI, Google, Meta, Verizon, Associated Press.
   - **Characteristics**: Highlights partnerships and developments in AI technologies, in

In [52]:
#@title Bonus — Self-Correction Graph Retrieval (hop 2 -> hop 3 -> vector fallback)
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# Demo trên 2 câu golden (1 multi-hop, 1 cross-doc) — có stop condition rõ ràng
for _, _row in golden_df[golden_df.group.isin(["multi-hop","cross-doc"])].groupby("group").head(1).iterrows():
    _sc = self_correcting_context(_row.question)
    print(f"[{_row.id} · {_row.group}] route={_sc['route']}")
    if _sc["missing"]:
        print("  missing:", _sc["missing"][:180])
    _ans = generate_answer(_row.question, _sc["context"])
    print("  answer:", _ans["answer"][:400].replace("\n", " "))
    print()


[G5000-01 · multi-hop] route=hop2
  answer: Aeris acquired Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses on January 18, 2023, as part of a transaction that involved the IoT business from Ericsson [chunk_id=art_000836::c0000]. This acquisition was initially announced on January 10, 2023, and was also mentioned in a prior report on December 7, 2022, indicating a partnership and the intent to acquire the IoT business [chunk

[G5000-02 · cross-doc] route=hop2
  answer: The first two Aeris/Ericsson reports describe a planned transfer, specifically Aeris's intention to acquire Ericsson's IoT business. The first report dated January 10, 2023, states that Aeris is to acquire the IoT business from Ericsson, indicating a planned acquisition. The second report dated January 18, 2023, confirms that the acquisition has been completed, as it states that Aeris has acquired



# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau